# Dynamic programming (DP)

Idea of breaking down a larger problems recursively into subproblems

Within context of RL, it essentially means turning the Bellman equation into an update rule and computing action-value functions using this.

Requires a perfect model of env.

All methods in RL are approximations of DP


##### Policy evaluation (Prediction)

How to compute action-value function

From previous section we know:

<img src="../image-66.png" width="500px" /><div/>


State-value function is the average of all the available action-values, weighted by how likely we are to choose them under our current policy:

We apply the Bellmann equation as an update rule, and compute for each state

<img src="../image-67.png" width="500px" /><div/>



<img src="../image-68.png" width="500px" /><div/>

###### Code implementation

Bellman equation to update V: 

$V(s)\leftarrow r(s)+\gamma\sum_{s'}P(s'|s)V(s')$

In [1]:
import numpy as np

# ---------- MDP definition (dicts) ----------

S = list(state_names)  # ["C1", "C2", "C3", "FB", "Sleep"]
terminal_states_mdp = {"Sleep"}

A = {
    "C1": ["study", "facebook"],
    "C2": ["study", "sleep"],
    "C3": ["study", "pub"],
    "FB": ["quit", "facebook"],
    "Sleep": [],
}

# Transition model: P[(s,a)] = [(prob, s_next), ...]
P_sa = {
    ("C1", "study"): [(1.0, "C2")],
    ("C1", "facebook"): [(1.0, "FB")],
    ("C2", "study"): [(1.0, "C3")],
    ("C2", "sleep"): [(1.0, "Sleep")],
    # Study in C3 gives +10 then day ends
    ("C3", "study"): [(1.0, "Sleep")],
    # Pub in C3 transitions back to class states
    ("C3", "pub"): [(0.2, "C1"), (0.4, "C2"), (0.4, "C3")],
    ("FB", "quit"): [(1.0, "C1")],
    ("FB", "facebook"): [(1.0, "FB")],
}

# Reward model: R[(s,a)] = immediate reward for taking action a in s
R_sa = {
    ("C1", "study"): -2.0,
    ("C1", "facebook"): -1.0,
    ("C2", "study"): -2.0,
    ("C2", "sleep"): 0.0,
    ("C3", "study"): 10.0,
    ("C3", "pub"): 1.0,
    ("FB", "quit"): 0.0,
    ("FB", "facebook"): -1.0,
}


def actions(s):
    return A[s]


def transitions_sa(s, a):
    return P_sa[(s, a)]


def reward_sa(s, a):
    return float(R_sa[(s, a)])


# ---------- Bellman operators ----------


def bellman_expectation_backup_v(V, pi, gamma):
    """Bellman expectation equation for V under policy pi.

    V(s) = Σ_a π(a|s) Σ_{s'} P(s'|s,a) [ r(s,a) + γ V(s') ]
    """
    V_new = {}
    for s in S:
        if s in terminal_states_mdp:
            V_new[s] = 0.0
            continue
        v = 0.0
        for a, p_a in pi[s].items():
            q_sa = 0.0
            for p, s_next in transitions_sa(s, a):
                q_sa += p * (reward_sa(s, a) + gamma * V[s_next])
            v += p_a * q_sa
        V_new[s] = v
    return V_new


def q_from_v(V, gamma):
    """Compute Q(s,a) from V: Q(s,a) = Σ_{s'} P(s'|s,a) [ r(s,a) + γ V(s') ]."""
    Q = {s: {} for s in S}
    for s in S:
        if s in terminal_states_mdp:
            continue
        for a in actions(s):
            q = 0.0
            for p, s_next in transitions_sa(s, a):
                q += p * (reward_sa(s, a) + gamma * V[s_next])
            Q[s][a] = q
    return Q


# ---------- Dynamic programming algorithms ----------


def normalize_policy(pi):
    """Ensure pi[s] is a valid distribution over actions(s)."""
    out = {}
    for s in S:
        if s in terminal_states_mdp:
            out[s] = {}
            continue
        if s not in pi or not pi[s]:
            # default uniform
            acts = actions(s)
            out[s] = {a: 1.0 / len(acts) for a in acts}
            continue

        # keep only valid actions
        probs = {a: float(p) for a, p in pi[s].items() if a in set(actions(s))}
        total = sum(probs.values())
        if total <= 0:
            acts = actions(s)
            out[s] = {a: 1.0 / len(acts) for a in acts}
        else:
            out[s] = {a: p / total for a, p in probs.items()}
    return out


NameError: name 'state_names' is not defined

In [ ]:
def policy_evaluation(pi, gamma=1.0, theta=1e-12, max_iters=100_000):
    """Iterative policy evaluation using the Bellman expectation backup."""
    pi = normalize_policy(pi)
    V = {s: 0.0 for s in S}

    for _ in range(max_iters):
        V_new = bellman_expectation_backup_v(V, pi, gamma)
        delta = max(abs(V_new[s] - V[s]) for s in S)
        V = V_new
        if delta < theta:
            break

    Q = q_from_v(V, gamma)
    return V, Q


# Uniform random policy (matches the earlier MDP->MRP conversion)
pi_uniform = {s: {a: 1.0 / len(actions(s)) for a in actions(s)} for s in S}

V_pi, Q_pi = policy_evaluation(pi_uniform, gamma=1.0, theta=1e-12)
print("V^pi (uniform random):")
for s in S:
    print(f"  {s:>5}: {V_pi[s]: .6f}")


V^pi (uniform random):
     C1: -1.307692
     C2:  2.692308
     C3:  7.384615
     FB: -2.307692
  Sleep:  0.000000


In [ ]:
def value_iteration(gamma=1.0, theta=1e-12, max_iters=100_000):
    """Value iteration using the Bellman optimality backup.

    V(s) = max_a Σ_{s'} P(s'|s,a)[ r(s,a) + γ V(s') ]
    """
    V = {s: 0.0 for s in S}

    for _ in range(max_iters):
        delta = 0.0
        V_new = dict(V)
        for s in S:
            if s in terminal_states_mdp:
                V_new[s] = 0.0
                continue
            best = -float("inf")
            for a in actions(s):
                q = 0.0
                for p, s_next in transitions_sa(s, a):
                    q += p * (reward_sa(s, a) + gamma * V[s_next])
                best = max(best, q)
            delta = max(delta, abs(best - V[s]))
            V_new[s] = best
        V = V_new
        if delta < theta:
            break

    Q = q_from_v(V, gamma)

    # Greedy deterministic policy from Q*
    pi_star = {}
    for s in S:
        if s in terminal_states_mdp:
            pi_star[s] = {}
            continue
        best_a = max(Q[s].items(), key=lambda kv: kv[1])[0]
        pi_star[s] = {a: (1.0 if a == best_a else 0.0) for a in actions(s)}

    return V, Q, pi_star


# ---------- Run on the same example ----------

V_star, Q_star, pi_star = value_iteration(gamma=1.0, theta=1e-12)
print("\nV* (optimal):")
for s in S:
    print(f"  {s:>5}: {V_star[s]: .6f}")



V* (optimal):
     C1:  6.000000
     C2:  8.000000
     C3:  10.000000
     FB:  6.000000
  Sleep:  0.000000


In [ ]:
print("\nGreedy optimal policy π*:")
for s in S:
    if s in terminal_states_mdp:
        continue
    best_a = max(pi_star[s].items(), key=lambda kv: kv[1])[0]
    print(f"  {s:>5} -> {best_a}")

print("\nQ* (optimal action-values):")
for s in S:
    if s in terminal_states_mdp:
        continue
    items = ", ".join(f"{a}={Q_star[s][a]:.6f}" for a in actions(s))
    print(f"  {s:>5}: {items}")



Greedy optimal policy π*:
     C1 -> study
     C2 -> study
     C3 -> study
     FB -> quit

Q* (optimal action-values):
     C1: study=6.000000, facebook=5.000000
     C2: study=8.000000, sleep=0.000000
     C3: study=10.000000, pub=9.400000
     FB: quit=6.000000, facebook=5.000000


In [ ]:
def iterative_policy_evaluation(
    states,
    transitions,
    rewards,
    terminal_states,
    gamma=1.0,
    theta=1e-10,
    max_iters=100_000,
):
    """Iterative policy evaluation for V under a fixed (implicit) policy.

    Here, the transition model already reflects the policy (MRP-style), so we evaluate:
    V(s) = r(s) + gamma * sum_{s'} P(s'|s) * V(s')
    """
    V = {s: 0.0 for s in states}

    # Keep terminal values fixed (single terminal reward, then stop).
    for t in terminal_states:
        V[t] = float(rewards[t])

    deltas = []

    for i in range(max_iters):
        delta = 0.0

        for s in states:
            if s in terminal_states:
                continue

            v_old = V[s]
            exp_next = sum(p * V[s_next] for p, s_next in transitions[s])
            V[s] = float(rewards[s]) + gamma * exp_next
            delta = max(delta, abs(v_old - V[s]))

        deltas.append(delta)
        if delta < theta:
            print(f"Converged after {i + 1} iterations (delta={delta:.3e})")
            break

    return V, deltas


V_eval, deltas = iterative_policy_evaluation(
    states=states,
    transitions=transitions_with_terminal_states,
    rewards=r,
    terminal_states=terminal_states,
    gamma=1.0,
    theta=1e-10,
)

print("\nEstimated V under the current policy:")
for s in states:
    print(f"  {s:>10}: {V_eval[s]: .6f}")
print("\nAnalytical Solution:")
for state in range(len(states)):
    print(f"  {states[state]:>10}: {analytical_solution[state]: .6f}")

Converged after 479 iterations (delta=9.637e-11)

Estimated V under the current policy:
      Class1: -12.543210
      Class2:  1.456790
      Class3:  4.320988
    Facebook: -22.543210
         Pub:  0.802469
       Sleep:  0.000000
        Pass:  10.000000

Analytical Solution:
      Class1: -12.543210
      Class2:  1.456790
      Class3:  4.320988
    Facebook:  10.000000
         Pub:  0.802469
       Sleep: -22.543210
        Pass:  0.000000


#### Policy improvement

From predicting value (estimation) -> move to control problem



Similarly, the action-value function is the average value of the states the agent might transition to given the chosen action, but weighted instead by their transition probabilities

<img src="../image-69.png" width="500px" /><div/>

<img src="../image-44.png" width="600px" /><div/>

We do a one step policy improvement

#### Backup diagram

<img src="../image-75.png" width="600px" /><div/>




So action-value is the immediate reward for taking that action, plus the expected value of the next state, over all possible successor states

If this is better than the value of π, we are better off changing our policy in such a way. And we can indeed answer such question via the policy improvement theorem, which states that if, for all states s:

<img src="../image-70.png" width="200px" /><div/>

The improved policy is better:

<img src="../image-71.png" width="200px" /><div/>




### Policy iteration

chain these improvement steps with policy evaluation


<img src="../image-72.png" width="700px" /><div/>

Each of the resulting policies are monotonically improving, and — since finite MDPs only have finite states — this process must converge to the optimal policy eventually!

So first algorithm for solving an RL problem — called policy iteration


<img src="../image-73.png" width="700px" /><div/>
